# 01. 디코딩 전략 실습 — Transformers

**목표**: 디코딩 파라미터가 LLM 출력에 미치는 영향을 직접 관찰

**다루는 것**
- Greedy / Sampling / Beam의 출력 차이
- temperature·top_k·top_p가 다음 토큰 확률 분포를 어떻게 잘라내는지 시각화

**다루지 않는 것**
- API 서버와 앱 연결은 03번 이후의 실험에서 다룹니다

**실습 환경**: 로컬 RTX A4000 (16GB VRAM) · 모델 `Qwen/Qwen3-4B-Instruct-2507` (bf16 ≈ 8GB, non-thinking 전용 Instruct 모델)

> 이 노트북은 HF transformers로 모델을 **직접** GPU에 올립니다. llama-server 등 다른 GPU 프로세스는 필요 없습니다.

In [ ]:
import ipywidgets as widgets
print('ipywidgets:', widgets.__version__)

ipywidgets: 8.1.9


## 🧭 시작 전에

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | LLM이 토큰을 하나씩 고르는 원리(확률분포 → 디코딩)를 이해하고, temperature·top-k·top-p·beam이 답변을 어떻게 바꾸는지 직접 본다 |
| 📚 사전 지식 | 00번 노트북 (토큰, 파라미터, VRAM 개념) |
| 🐍 커널(환경) | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | Qwen3-4B 원본(bf16) ≈ 8~9 GB |
| ⏱️ 소요 시간 | 약 40분 (처음에는 모델 다운로드 ≈ 8GB, +5~10분) |
| ✔️ 실행 전 확인 | 다른 노트북 커널 종료, `python local_server.py stop` (서버가 켜져 있다면) |

> **실행 요령**: 위에서부터 **한 셀씩**(Shift+Enter) 실행하세요. `Run All`은 쓰지 마세요.
> `✅ TODO 셀`은 학생이 채우는 칸이라 그대로 실행하면 일부러 에러가 납니다. 막히면 바로 아래 `✅ 참고 답안` 셀을 보세요.

## 📖 핵심 개념: LLM은 어떻게 문장을 만들까?

```
① 토크나이저: "대한민국의 수도는" → [토큰ID 1234, 567, 89]   (글자 → 숫자)
② 모델:       토큰ID들 → 다음 토큰 후보 15만 개 각각의 점수(logits)
③ softmax:    점수 → 확률  (" 서울" 0.92, " 세종" 0.01, ...)
④ 디코딩:     확률분포에서 토큰 하나 선택  ← 이 노트북의 주제!
⑤ 선택한 토큰을 입력 끝에 붙이고 ②로 돌아가 반복 (종료 토큰이 나오거나 max_new_tokens에 도달할 때까지)
```

| 디코딩 방법 | 동작 | 특징 |
|---|---|---|
| **Greedy** | 항상 확률 1등 선택 | 매번 같은 답. 안정적이지만 단조롭고 반복될 수 있음 |
| **Sampling** | 확률에 비례해 무작위 선택 | 매번 다른 답. 자연스럽고 다양함 |
| **Beam search** | 후보 문장 여러 개를 동시에 키워 전체 점수가 가장 높은 문장 선택 | 매끄럽지만 느림 |
| **temperature** | 확률분포를 뾰족하게(<1) 또는 평평하게(>1) 조절 | "창의성 다이얼" |
| **top-k / top-p** | 확률 낮은 꼬리 후보를 잘라냄 | 엉뚱한 단어 방지 |

> 여기서 배우는 `temperature`는 05~09번에서 `get_chat_model(temperature=0)`처럼 그대로 다시 등장합니다.

## 0. 환경 설정

**아래 셀이 하는 일**
1. 그래프의 한글이 깨지지 않도록 한글 폰트를 내려받아 설정 (처음 1회, 인터넷 필요)
2. Hugging Face에서 `Qwen/Qwen3-4B-Instruct-2507` 모델을 내려받아 GPU에 로드 (처음 1회 약 8GB, `~/.cache/huggingface`에 저장)
   - `AutoTokenizer`: 글자 ↔ 토큰ID 변환기
   - `AutoModelForCausalLM`: 다음 토큰을 예측하는 언어 모델 본체
   - `dtype=bfloat16`: 파라미터를 2바이트로 저장 (A4000이 지원하는 효율적 형식)
   - `device_map='auto'`: GPU에 자동 배치

In [10]:
# 패키지는 setup_environment.bat으로 설치합니다. 환경을 다시 맞출 때 setup_environment.bat을 실행하세요.
import torch, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

# ── 한글 폰트 설정 ──
_font_path = Path.home() / '.cache' / 'fonts' / 'NotoSansKR-Regular.ttf'
if not _font_path.exists():
    import urllib.request
    _font_path.parent.mkdir(parents=True, exist_ok=True)
    _url = 'https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf'
    print(f'Downloading Korean font from {_url} ...')
    urllib.request.urlretrieve(_url, str(_font_path))
    print('Done.')
fm.fontManager.addfont(str(_font_path))
plt.rcParams['font.family'] = fm.FontProperties(fname=str(_font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 부호 깨짐 방지

# A4000(16GB)에서 부담 없는 4B Instruct 모델 (bf16 ≈ 8GB, beam search 4개까지 여유)
# 더 가볍게 돌리고 싶다면: 'Qwen/Qwen3-1.7B' (단, thinking 모델이므로 make_inputs에서 enable_thinking=False 필요)
MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# A4000(Ampere)은 bf16 지원 → fp16보다 수치적으로 안정적
if device == 'cuda':
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map='auto' if device == 'cuda' else None,
).eval()
print(f'Loaded {MODEL_ID} on {device} ({DTYPE})')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded Qwen/Qwen3-4B-Instruct-2507 on cuda (torch.bfloat16)


In [ ]:
def make_inputs(prompt: str):
    """Apply chat template and tokenize. 모든 셀에서 공통으로 사용."""
    msgs = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors='pt').to(model.device)

def gen(prompt: str, **kwargs) -> str:
    """한 번 생성하고 응답 부분만 반환."""
    inputs = make_inputs(prompt)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, **kwargs)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# 빠른 동작 확인
print(gen('한 문장으로 자기소개해 줘.', do_sample=False))

tensor([[151644,    872,    198,  23573,  53435,  40853,  42039, 131823,  43590,
          59761,  33883,  53989,    246,     13, 151645,    198, 151644,  77091,
            198, 126781,  16560, 131502,  20401, 138520,  19391,  81173, 125519,
          17877,  49367,  33883, 129392,  80573, 134794,  58677,  78125,  21329,
          66019,  78952,     13, 151645]], device='cuda:0')
저는 당신의 질문에 최선을 다해 도와주는 인공지능입니다.


### 🔍 들여다보기: 토큰과 chat template

`make_inputs()` 안에서 쓴 `apply_chat_template`은 대화 메시지를 모델이 **학습할 때 본 형식**으로 바꿔 줍니다.
Instruct 모델은 이 형식이어야 "질문에 답하는 모드"로 동작합니다. 실제로 어떻게 생겼는지 확인해 봅시다.

In [13]:
# 1) chat template 적용 결과 — 모델이 실제로 읽는 텍스트
msgs = [{'role': 'system', 'content': '당신은 친절한 비서입니다.'},
        {'role': 'user', 'content': '안녕하세요'}]
print(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

# 2) 토큰화 — 글자가 어떤 조각(토큰)으로 쪼개지는지
text = '대한민국의 수도는 서울입니다. Hello world!'
ids = tokenizer(text)['input_ids']
print('토큰 수:', len(ids))
print('토큰ID :', ids)
print('조각   :', [tokenizer.decode([i]) for i in ids])
print('어휘 크기(vocab):', len(tokenizer))

<|im_start|>system
당신은 친절한 비서입니다.<|im_end|>
<|im_start|>user
안녕하세요<|im_end|>
<|im_start|>assistant

토큰 수: 13
토큰ID : [66845, 23573, 125496, 124785, 20401, 134013, 16560, 136905, 78952, 13, 21927, 1879, 0]
조각   : ['대', '한', '민', '국', '의', ' 수도', '는', ' 서울', '입니다', '.', ' Hello', ' world', '!']
어휘 크기(vocab): 151669


## 1. Greedy vs Sampling — 같은 프롬프트, 다른 출력

Greedy는 매번 같은 출력. Sampling은 매번 다른 출력.

운영 환경에서 "왜 답변이 매번 달라요?" 라는 질문은 보통 sampling이 켜져 있어서 발생함. 어떤 시나리오에 어떤 디코딩이 맞는지 판단하는 감각이 필요.

### TODO #1 — Greedy 5회 vs Sampling 5회

프롬프트 하나로 다음 두 가지를 각각 5번씩 생성하고 출력하시오.

1. **Greedy**: `do_sample=False`
2. **Sampling**: `do_sample=True, temperature=1.0, top_p=0.95`

관찰할 것: Greedy는 5회가 모두 같은가? Sampling 5회의 *체감* 다양성은 어느 정도인가?

In [16]:
PROMPT = '서울에서 데이트하기 좋은 장소를 한 곳 추천해 줘. 이유도 한 줄로.'

print('=== Greedy ===')
for i in range(5):
    # TODO: Greedy 디코딩(do_sample=False)으로 1회 생성해 result 에 저장
    result = gen(PROMPT, do_sample=False)
    print(f'[{i+1}] {result}')

print('\n=== Sampling (temp=1.0, top_p=0.95) ===')
for i in range(5):
    # TODO: Sampling 디코딩(do_sample=True, temperature=1.0, top_p=0.95)으로 1회 생성
    result = None
    print(f'[{i+1}] {result}')

=== Greedy ===
tensor([[151644,    872,    198,  26698, 126893,  56475,   5140,  46218,  28626,
          66425, 132968, 129359,  43590,  18411,  61298,  45130,    111,  57835,
         129034,  33883,  53989,    246,     13, 132819,  47985,  61298,  53989,
            226,  17380,     13, 151645,    198, 151644,  77091,    198,  26698,
         126893,  56475,   5140,  46218,  28626,  66425, 132968, 129359,  43590,
             25,   3070,  26698, 126893, 144438,    334,   2303,  12802, 125522,
             25, 140577,  53680, 129392,  29326,  19969,  65510,  56290,  18411,
         139646, 124905,  10764,    236,    116, 126246,  23573, 128618,  80901,
          20487,  80573, 129676, 127165,    108, 126712,     11, 139764,     11,
          10764,    240,    235,  65306, 129423,  55902,  12802,  95351,  33883,
         132488,  24897,  60294,  93672,   5140,  46218,  28626,  19391,  81173,
          80968,  78952,     13, 151645]], device='cuda:0')
[1] 서울에서 데이트하기 좋은 장소: **서울숲**  
이유

In [6]:
# ✅ 참고 답안 (핵심) — 막힐 때만
result = gen(PROMPT, do_sample=False)                              # Greedy: 5회 모두 동일
result = gen(PROMPT, do_sample=True, temperature=1.0, top_p=0.95)  # Sampling: 매번 다름

**관찰 포인트**
- Greedy 5회: 완전히 동일한 출력이 반복됨 — 결정론적(deterministic) 디코딩이기 때문.
- Sampling 5회: 매번 다른 장소/이유가 나올 수 있음. temperature가 높을수록, top_p가 넓을수록 다양성 증가.
- 운영 환경에서 재현성이 필요하다면 `do_sample=False` (또는 `seed` 고정) 필수.

## 2. Temperature 스윕 — "창의성" 다이얼

Temperature `T`는 softmax 직전에 logits를 `T`로 나눈다. `T < 1`이면 분포가 sharp해지고, `T > 1`이면 flat해짐. `T = 0`은 사실상 greedy.

수학적으로:
$$P(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

### TODO #2 — Temperature 스윕

같은 프롬프트로 `temperature ∈ {0.0, 0.3, 0.7, 1.2}` 각각 1회 생성. 어느 쪽이 "답이 헛소리에 가까워지는" 임계점인지 본인 눈으로 확인.

**힌트**: `temperature=0.0`은 transformers에서 에러를 낼 수 있음. 그 경우 `do_sample=False`로 처리.

In [7]:
PROMPT = '한국 대기업의 향후 5년 AI 전략을 3가지로 요약해 줘.'

for T in [0.0, 0.3, 0.7, 1.2]:
    print(f'\n--- temperature = {T} ---')
    # TODO: temperature=T 로 1회 생성 (T=0.0 이면 do_sample=False, 아니면 do_sample=True)
    result = None
    print(result)


--- temperature = 0.0 ---
None

--- temperature = 0.3 ---
None

--- temperature = 0.7 ---
None

--- temperature = 1.2 ---
None


In [8]:
# ✅ 참고 답안 (핵심)
if T == 0.0:
    result = gen(PROMPT, do_sample=False)            # T=0 은 greedy 로 처리(에러 회피)
else:
    result = gen(PROMPT, do_sample=True, temperature=T)

**관찰 포인트**
- `T=0.0` (greedy): 가장 확실하지만 단조로운 답변.
- `T=0.3`: 여전히 사실 중심, 약간의 표현 다양성.
- `T=0.7`: 자연스러운 다양성, 실용적 sweet spot.
- `T=1.2`: 창의적이지만 일관성 저하, 문장이 이상해질 수 있음.

## 3. top_k vs top_p 시각화

두 방식 모두 "확률 낮은 후보를 잘라내고 나머지에서 sampling"이지만 자르는 방식이 다름.

- **top_k**: 확률 상위 `k`개만 남김 (고정 개수)
- **top_p (nucleus)**: 누적 확률이 `p`를 넘기까지의 토큰만 남김 (가변 개수)

어느 한 토큰 위치에서 실제 분포를 뽑아 *얼마나 다르게 자르는지* 확인.

In [15]:
# 한 시점의 next-token 분포를 가져옴
PROMPT = '대한민국의 수도는'
inputs = tokenizer(PROMPT, return_tensors='pt').to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]  # (vocab_size,)
probs = torch.softmax(logits, dim=-1).cpu().numpy()

# 상위 30개만 가시화
TOP_VIEW = 30
top_idx = np.argsort(probs)[::-1][:TOP_VIEW]
top_probs = probs[top_idx]
top_tokens = [tokenizer.decode([int(i)]) for i in top_idx]
print('상위 10개 후보:')
for tok, p in zip(top_tokens[:10], top_probs[:10]):
    print(f'  {tok!r:20s} {p:.4f}')

TypeError: Got unsupported ScalarType BFloat16

### TODO #3 — top_k와 top_p가 자르는 토큰 시각화

위 30개 후보에 대해, 다음 두 마스크를 만들고 막대그래프로 비교하시오.

1. `keep_topk`: 상위 `k=5`개만 True
2. `keep_topp`: 누적 확률이 `p=0.9`를 처음 넘는 시점까지 True

막대그래프: 살아남은 토큰은 색칠, 잘려나간 토큰은 회색으로 표시. 두 방식이 같은 후보를 자르는가? 다르다면 왜?

**구현 힌트**:
- top_k: `keep_topk = np.zeros(TOP_VIEW, dtype=bool); keep_topk[:k] = True`
- top_p: 누적합 `cumsum`을 활용. 첫 인덱스가 `p`를 넘는 위치까지 포함.

In [ ]:
k = 5
p = 0.9

# TODO: keep_topk 마스크 — 상위 k개만 True (shape=(TOP_VIEW,), dtype=bool)
keep_topk = None

# TODO: keep_topp 마스크 — 누적 확률이 p를 처음 넘는 시점까지 True
keep_topp = None

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, mask, title in zip(axes,
                           [keep_topk, keep_topp],
                           [f'top_k = {k}', f'top_p = {p}']):
    colors = ['#1f77b4' if m else '#cccccc' for m in mask]
    ax.bar(range(TOP_VIEW), top_probs, color=colors)
    ax.set_xticks(range(TOP_VIEW))
    ax.set_xticklabels([t.strip()[:6] for t in top_tokens], rotation=75, fontsize=7)
    ax.set_title(title)
    ax.set_ylabel('P(token)')
plt.tight_layout(); plt.show()
print(f'top_k 채택: {keep_topk.sum()}개,  top_p 채택: {keep_topp.sum()}개')

In [ ]:
# ✅ 참고 답안 (핵심)
keep_topk = np.zeros(TOP_VIEW, dtype=bool); keep_topk[:k] = True
cumsum = np.cumsum(top_probs)              # top_probs 는 내림차순 정렬됨
cutoff = np.searchsorted(cumsum, p)        # cumsum >= p 인 첫 위치
keep_topp = np.zeros(TOP_VIEW, dtype=bool); keep_topp[:cutoff + 1] = True

**관찰 포인트**
- `top_k=5`는 항상 정확히 5개만 선택 (분포 형태와 무관).
- `top_p=0.9`는 분포 형태에 따라 채택 개수가 달라짐:
  - 1위 토큰이 0.9 이상의 확률을 가진다면 단 1개만 선택.
  - 분포가 flat하면 더 많은 토큰이 포함됨.
- 이 예시('대한민국의 수도는')처럼 특정 토큰에 확률이 집중되면 top_p가 훨씬 적은 수를 채택할 수 있음.

## 4. 종합 비교

지금까지 본 디코딩 전략들을 *동일한 프롬프트*에 적용해 한눈에 비교.

### 선택 실습 — 5가지 전략으로 같은 질문 답변 비교 (시간이 남으면)

다음 5가지 설정으로 같은 프롬프트에 대해 한 번씩 생성, 출력의 길이 / 톤 / 사실성 / 반복 여부를 비교 후 짧은 코멘트.

1. Greedy: `do_sample=False`
2. Beam search: `num_beams=4, do_sample=False`
3. Sampling: `do_sample=True, temperature=0.7, top_p=0.9`
4. High-temp sampling: `do_sample=True, temperature=1.5`
5. Greedy + repetition penalty: `do_sample=False, repetition_penalty=1.3`

In [ ]:
PROMPT = '엘리베이터에서 만난 면접관에게 30초 자기소개를 해주세요.'

configs = {
    'greedy':        dict(do_sample=False),
    'beam_4':        dict(num_beams=4, do_sample=False),
    'sampling':      dict(do_sample=True, temperature=0.7, top_p=0.9),
    'high_temp':     dict(do_sample=True, temperature=1.5),
    'greedy_reppen': dict(do_sample=False, repetition_penalty=1.3),
}

for name, kwargs in configs.items():
    print(f'\n=== [{name}] ===')
    # TODO: 각 설정(kwargs)으로 1회 생성해 출력
    result = None
    print(result)

In [ ]:
# ✅ 참고 답안 (핵심)
result = gen(PROMPT, **kwargs)   # configs 의 각 디코딩 설정을 그대로 전달

**관찰 포인트 / 코멘트**

| 전략 | 길이 | 톤 | 반복 | 사실성 |
|------|------|----|------|--------|
| greedy | 중간 | 정형적 | 있을 수 있음 | 높음 |
| beam_4 | 중간~길게 | 정형적·매끄러움 | 낮음 | 높음 |
| sampling (T=0.7, p=0.9) | 다양 | 자연스러움 | 낮음 | 높음 |
| high_temp (T=1.5) | 다양 | 산만·창의적 | 낮음 | 낮을 수 있음 |
| greedy + rep_penalty | 중간 | 정형적 | 매우 낮음 | 높음 |

- **Beam search**는 greedy보다 전체 시퀀스 점수를 최적화하므로 더 유창한 경향이 있음.
- **High temperature**는 다양하지만 문장이 어색해지거나 논점을 벗어날 수 있음.
- **Repetition penalty**는 greedy의 반복 문제를 완화하는 실용적인 방법.



## 정리

- **Greedy / Beam**: 같은 입력 → 같은 출력. 결정성 필요할 때.
- **(Temperature-based) Sampling**: 다양성. temperature가 "얼마나 과감하게" 다이얼.
- **top_k / top_p**: 꼬리 잘라내기. top_p가 분포 형태에 더 적응적.

## 다음 단계

모델 실험을 마쳤다면 커널을 종료해 GPU 메모리를 비우세요. API 기반 실험이 필요할 때 03번 노트북으로 이동합니다.

## ✅ 정리 & 스스로 점검

**정리**
- LLM은 매 단계 **다음 토큰의 확률분포**를 만들고, 디코딩 방법이 그중 하나를 고릅니다.
- 같은 모델이라도 디코딩 설정에 따라 답변 성격이 크게 달라집니다.

**상황별 추천 설정 (05~09 앱 실습에도 그대로 적용)**

| 상황 | 추천 |
|---|---|
| JSON 파싱, 도구 호출, 분류 (정확성·재현성 중요) | `temperature=0` (greedy) |
| 일반 챗봇 답변 | `temperature≈0.7, top_p≈0.9` |
| 아이디어·창작 | `temperature 0.9~1.0` |
| 반복이 심할 때 | `repetition_penalty 1.1~1.3` |

**스스로 점검**
1. temperature를 0에 가깝게 하면 왜 매번 같은 답이 나올까요?
2. top-k=5와 top-p=0.9 중 분포 모양에 따라 후보 수가 달라지는 것은 어느 쪽인가요?
3. "대한민국의 수도는" 다음 토큰 분포에서 1등 확률이 매우 높았다면, temperature를 올려도 답이 잘 안 바뀌는 이유는?

**다음 단계 →** API 기반 앱 실험이 필요하면 커널을 종료한 뒤 03번 Windows 노트북으로 이동하세요.